# Run an energy demand forecast from a notebook

Press play to run a real data job on Pipekit: aggregate synthetic half-hourly demand into daily totals and forecast the next day, per region. No Kubernetes, no YAML, no `kubectl`.

The platform team owns `dataplatform/` (image, resources, service account, namespace, connection). You write a single job and submit it.

## Prerequisites

Install the Pipekit CLI, then log in once. This writes a token to `~/.pipekit/token` that the SDK reuses, so you do not paste a token into the notebook.

```bash
brew install pipekit/tap/cli   # or see https://docs.pipekit.io/cli
pipekit login
```

This repo ships a Nix dev shell (`shell.nix` at the repo root) that provides Python and the C++ runtime the Jupyter kernel needs on NixOS. From the repo root, inside the dev shell:

```bash
direnv allow                 # or run: nix-shell
python -m venv .venv
.venv/bin/pip install hera pipekit-sdk ipykernel jupyter
```

On NixOS the kernel needs `libstdc++` on `LD_LIBRARY_PATH`, or `pyzmq` fails to load and the editor reports that `jupyter and notebook` are missing. Bake that path into the kernel so it works however you launch your editor:

```bash
.venv/bin/python -m ipykernel install --user --name hera-forecast \
  --display-name "Python (hera-forecast)" \
  --env LD_LIBRARY_PATH "$LD_LIBRARY_PATH"
```

Then select the `Python (hera-forecast)` kernel for this notebook (kernel picker, top right), not a bare `Python 3.x` interpreter. Re-run the install command after a nixpkgs upgrade, since the baked path points into the Nix store.

pandas and numpy run on the cluster, not locally, so you do not need them here.

## Setup

In [ ]:
import os
import sys

# VSCode starts the kernel at the workspace root, not this folder. Point the
# working dir at this example so `dataplatform` and `forecast_step` import.
nb = globals().get('__vsc_ipynb_file__')
here = os.path.dirname(nb) if nb else os.getcwd()
if not os.path.isdir(os.path.join(here, 'dataplatform')):
    here = os.path.join(os.getcwd(), 'examples', 'hera-notebook-forecast')
os.chdir(here)
sys.path.insert(0, here)
print('working dir:', os.getcwd())

## Point at your cluster

Set the name of your free trial cluster. If you used a different name when you provisioned it, change it here. The SDK targets `https://api.pipekit.io` by default, which is correct for the free trial; set `PIPEKIT_URL` only for a different environment.

In [ ]:
os.environ['PIPEKIT_CLUSTER'] = 'free-trial-cluster'

## What you write

A single function. `forecast_demand` (in `forecast_step.py`) is a Hera `@script` step that aggregates demand and forecasts the next day with pandas. The platform image, resources, service account, and namespace come from `dataplatform`; you never set them. `run` submits to Pipekit and prints a link to watch it live.

In [ ]:
from dataplatform import run
from forecast_step import forecast_demand

result = run('daily-demand-forecast', forecast_demand)

## Watch it run

Open the link printed above to see the graph and live logs in the Pipekit UI. To pull the logs inline, run the next cell: it waits for the run to finish, then prints the step's output (the forecast tables). The first run can take a couple of minutes while the image pulls.

In [ ]:
from dataplatform import logs, wait

wait(result)   # block until the run finishes, so its logs exist
logs(result)   # the forecast tables the step printed

## The same job as YAML (the GitOps path)

`to_yaml` renders the manifest you would commit to a feature branch. Same Python, now a versioned artifact. Pipekit pins per-branch versions, so committing it does not overwrite anyone else's work.

In [ ]:
from dataplatform import to_yaml

print(to_yaml('daily-demand-forecast', forecast_demand))